# Spotify API Exploration

Explores the Spotify Web API via `SpotifyClient`, decomposing responses into Polars DataFrames.

**Prerequisites:**
- `.env` populated with `SPOTIFY_CLIENT_ID` and `SPOTIFY_CLIENT_SECRET`
- For playlist cells: run `uv run python scripts/spotify_auth.py` first to populate `SPOTIFY_REFRESH_TOKEN`

In [ ]:
import polars as pl
from spotify import SpotifyClient

client = SpotifyClient.from_env()

## Raw Response Structure

Inspect the raw JSON returned by Spotify's `/search` endpoint to understand the available fields.

In [ ]:
raw = client.get_raw_search("Sweet Home Chicago", limit=2)
raw["tracks"]["items"][0]

Top-level keys on each track object:

In [ ]:
list(raw["tracks"]["items"][0].keys())

Pagination metadata — useful for understanding result set size:

In [ ]:
{k: v for k, v in raw["tracks"].items() if k != "items"}

## Track Search as a DataFrame

`search_tracks()` parses the response into a flat Polars DataFrame.

In [ ]:
client.search_tracks("Sweet Home Chicago")

Selecting key display columns, ranked by Spotify popularity score:

In [ ]:
client.search_tracks("Sweet Home Chicago").select(
    "name", "artists", "album", "release_date", "popularity", "explicit"
).sort("popularity", descending=True)

## Multiple Queries

Combine results from several searches into a single DataFrame for comparison.

In [ ]:
pl.concat([
    client.search_tracks(q, limit=5).with_columns(pl.lit(q).alias("query"))
    for q in ["rolling stones", "arctic monkeys", "amy winehouse"]
]).select("query", "name", "artists", "popularity").sort("query", "popularity")

## Track Duration

`duration_ms` is returned in milliseconds. Convert to minutes for readability.

In [ ]:
client.search_tracks("Sweet Home Chicago").select(
    "name",
    "artists",
    (pl.col("duration_ms") / 60_000).round(2).alias("duration_min"),
    "popularity",
).sort("duration_min", descending=True)

## Playlist Exploration

> Requires `SPOTIFY_REFRESH_TOKEN` in `.env`. Run `uv run python scripts/spotify_auth.py` first.

In [ ]:
client.get_playlists()

In [ ]:
playlist_id = client.find_playlist_id("DJ Requests")
playlist_id

In [ ]:
client.get_playlist_tracks(playlist_id)